In [1]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

In [2]:
df_minseo = pd.read_csv("C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/regresssion/eda_master_v2_by_gu_year.csv")
df_cluster = pd.read_csv("C:/Users/User/OneDrive/바탕 화면/유영우/빅콘2026/notebooks/cluster_result.csv")
df_minseo

,year,gu,waste_total,waste_per_capita,food_waste_total,recycle_rate,total_pop,households,elderly_ratio,parcel_arrival_total,arrival_per_hh,living_pop_daily_avg,day_night_ratio,biz_total,food_accom_biz,illegal_dumping_count
0,2020,강남구,254317,694.9,242.6,65.553620,544055,234872,13.803016,NaN,NaN,1.932652e+07,1.328122,115054,3816,22467
1,2020,강동구,134696,368.0,110.4,67.471194,463998,196499,15.090367,NaN,NaN,1.210167e+07,0.915846,40978,4897,8282
2,2020,강북구,88989,243.1,68.7,60.028768,311569,145896,20.355684,NaN,NaN,7.291377e+06,0.897156,26711,2994,9747
3,2020,강서구,192797,526.8,144.1,69.431060,585901,266982,15.187549,NaN,NaN,1.300812e+07,0.948083,58788,8490,362
4,2020,관악구,146977,401.6,94.2,70.433469,509803,274811,15.471663,NaN,NaN,1.169594e+07,0.881875,38639,4050,6151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2023,용산구,208742,571.9,63.2,82.395014,227106,107825,17.501519,3634526.0,33.707637,6.765582e+06,1.160108,28363,1701,645
96,2023,은평구,123761,339.1,72.6,59.823369,470869,215721,19.924225,5192988.0,24.072705,1.006468e+07,0.855890,36201,4716,7025
97,2023,종로구,97419,266.9,79.9,56.079410,150453,72067,19.118263,2677305.0,37.150221,6.708106e+06,1.548435,46977,1916,20268
98,2023,중구,132481,363.0,101.7,59.428144,131793,64714,19.667205,3291386.0,50.860494,6.341925e+06,1.856137,67202,2325,9012


In [3]:
df_cluster

,gu,cluster_anchor,cluster_type,L1_work_home_ratio,L2_weekend_active_ratio,weekend_home_ratio,B1_avg_emp_per_biz,B2_office_emp_ratio_no_support,B3_consumption_tourism_emp_ratio,B4_life_service_emp_ratio,B5_industrial_logistics_emp_ratio,PC1_도시기능,PC2_도시기능
0,종로구,0,상업형,1.639698,0.795812,1.120581,5.672325,0.307603,0.270190,0.158138,0.132324,3.458148,1.960962
1,중구,0,상업형,2.064828,0.629787,1.083157,5.960833,0.410461,0.276900,0.074487,0.119097,5.194560,1.344587
2,용산구,0,상업형,1.184049,0.975406,1.076227,5.656782,0.311226,0.344603,0.141911,0.084112,1.909782,0.504961
3,성동구,3,혼합형,1.092607,0.911819,0.982540,5.119779,0.264219,0.266925,0.150575,0.206280,0.829732,-0.715104
4,광진구,2,유동집중형,0.937800,1.047381,0.982846,3.768584,0.143997,0.298187,0.273648,0.181917,-1.289669,0.229185
5,동대문구,2,유동집중형,0.982189,0.977135,0.971601,3.378678,0.124829,0.317755,0.284616,0.179222,-1.331756,0.346242
6,중랑구,2,유동집중형,0.847549,1.097254,0.967456,2.727116,0.076985,0.272712,0.268140,0.302640,-2.305739,0.312982
7,성북구,1,주거형,0.897531,1.021568,0.951521,3.531200,0.122407,0.262859,0.367205,0.155216,-2.065834,0.189028
8,강북구,2,유동집중형,0.873287,1.080638,0.976905,2.953737,0.107578,0.302779,0.294023,0.199744,-2.043262,0.443315
9,도봉구,1,주거형,0.866140,1.096668,0.978875,3.165744,0.118359,0.249941,0.315619,0.194613,-2.066692,0.451756


In [4]:
# 복사본 생성
reg = df_minseo.copy()
clu = df_cluster.copy()

# 자치구명 공백 제거
reg["gu"] = reg["gu"].astype(str).str.strip()
clu["gu"] = clu["gu"].astype(str).str.strip()

# 회귀에는 우선 cluster_type만 사용
# cluster_anchor는 숫자라서 같이 보관만 해도 됨
cluster_use = clu[["gu", "cluster_anchor", "cluster_type"]].drop_duplicates()

# merge
df = reg.merge(cluster_use, on="gu", how="left")

print(df.shape)
print("cluster_type 결측 수:", df["cluster_type"].isna().sum())

df[["gu", "cluster_anchor", "cluster_type"]].drop_duplicates().sort_values("gu")

(100, 18)
cluster_type 결측 수: 0


,gu,cluster_anchor,cluster_type
0,강남구,0,상업형
1,강동구,2,유동집중형
2,강북구,2,유동집중형
3,강서구,3,혼합형
4,관악구,1,주거형
5,광진구,2,유동집중형
6,구로구,3,혼합형
7,금천구,3,혼합형
8,노원구,1,주거형
9,도봉구,1,주거형


In [5]:
# 파생변수 만들기 
df_model = df.copy()

# 종속변수
df_model["log_waste_total"] = np.log(df_model["waste_total"])

# 규모 변수 로그
df_model["log_total_pop"] = np.log(df_model["total_pop"])
df_model["log_living_pop"] = np.log(df_model["living_pop_daily_avg"])

# 구조 변수
df_model["biz_per_10k"] = df_model["biz_total"] / df_model["total_pop"] * 10000
df_model["food_accom_ratio"] = df_model["food_accom_biz"] / df_model["biz_total"]
df_model["illegal_per_10k"] = df_model["illegal_dumping_count"] / df_model["total_pop"] * 10000

# 범주형 처리
df_model["year"] = df_model["year"].astype(int)
df_model["cluster_type"] = df_model["cluster_type"].astype("category")

# 확인
df_model[[
    "year", "gu", "cluster_type",
    "log_waste_total", "log_total_pop", "log_living_pop",
    "elderly_ratio", "day_night_ratio", "biz_per_10k",
    "food_accom_ratio", "illegal_per_10k"
]].head()

,year,gu,cluster_type,log_waste_total,log_total_pop,log_living_pop,elderly_ratio,day_night_ratio,biz_per_10k,food_accom_ratio,illegal_per_10k
0,2020,강남구,상업형,12.446337,13.206806,16.776989,13.803016,1.328122,2114.749428,0.033167,412.954573
1,2020,강동구,유동집중형,11.810776,13.047636,16.308854,15.090367,0.915846,883.150358,0.119503,178.492149
2,2020,강북구,유동집중형,11.396268,12.649376,15.802203,20.355684,0.897156,857.306086,0.112089,312.836001
3,2020,강서구,혼합형,12.169393,13.280906,16.381085,15.187549,0.948083,1003.377704,0.144417,6.178518
4,2020,관악구,주거형,11.898031,13.141780,16.274752,15.471663,0.881875,757.920216,0.104816,120.654449


In [6]:
#표준화 
model_cols = [
    "year", "gu", "cluster_type",
    "waste_total", 
    "log_waste_total",
    "log_total_pop",
    "log_living_pop",
    "elderly_ratio",
    "day_night_ratio",
    "biz_per_10k",
    "food_accom_ratio",
    "illegal_per_10k"
]

reg_data = df_model[model_cols].dropna().copy()

z_cols = [
    "log_total_pop",
    "log_living_pop",
    "elderly_ratio",
    "day_night_ratio",
    "biz_per_10k",
    "food_accom_ratio",
    "illegal_per_10k"
]

for col in z_cols:
    reg_data[col + "_z"] = (reg_data[col] - reg_data[col].mean()) / reg_data[col].std()

print(reg_data.shape)
print(reg_data["cluster_type"].value_counts())

(100, 19)
cluster_type
주거형      32
상업형      24
혼합형      24
유동집중형    20
Name: count, dtype: int64


In [7]:
# 기준모형을 주거형으로
reg_data["cluster_type"] = reg_data["cluster_type"].astype("category")

cats = list(reg_data["cluster_type"].cat.categories)
print(cats)

if "주거형" in cats:
    new_order = ["주거형"] + [c for c in cats if c != "주거형"]
    reg_data["cluster_type"] = reg_data["cluster_type"].cat.reorder_categories(
        new_order,
        ordered=False
    )

print(reg_data["cluster_type"].cat.categories)

['상업형', '유동집중형', '주거형', '혼합형']
Index(['주거형', '상업형', '유동집중형', '혼합형'], dtype='object')


In [8]:
# 기본 민서 변수만 
m1_formula = """
log_waste_total ~
    log_total_pop_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
"""

m1 = smf.ols(m1_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.603
Model:                            OLS   Adj. R-squared:                  0.558
Method:                 Least Squares   F-statistic:                     14.98
Date:                Thu, 07 May 2026   Prob (F-statistic):           5.09e-08
Time:                        10:52:04   Log-Likelihood:                -1.1367
No. Observations:                 100   AIC:                             24.27
Df Residuals:                      89   BIC:                             52.93
Df Model:                          10                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             11.8058      0

In [9]:
# 클러스터변수 추가 
m2_formula = """
log_waste_total ~
    log_total_pop_z
    + log_living_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
    + C(cluster_type)
"""

m2 = smf.ols(m2_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.684
Model:                            OLS   Adj. R-squared:                  0.637
Method:                 Least Squares   F-statistic:                     26.00
Date:                Thu, 07 May 2026   Prob (F-statistic):           5.54e-11
Time:                        10:52:05   Log-Likelihood:                 10.351
No. Observations:                 100   AIC:                             7.297
Df Residuals:                      86   BIC:                             43.77
Df Model:                          13                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [10]:
#M3. 클러스터 × 생활인구 interaction
m3_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + food_accom_ratio_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
"""

m3 = smf.ols(m3_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m3.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.740
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     25.19
Date:                Thu, 07 May 2026   Prob (F-statistic):           3.44e-11
Time:                        10:52:05   Log-Likelihood:                 19.969
No. Observations:                 100   AIC:                            -5.937
Df Residuals:                      83   BIC:                             38.35
Df Model:                          16                                         
Covariance Type:              cluster                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

In [11]:
#M4. 클러스터 × 생활인구 + 음식숙박업 interaction
m4_formula = """
log_waste_total ~
    log_total_pop_z
    + elderly_ratio_z
    + day_night_ratio_z
    + biz_per_10k_z
    + illegal_per_10k_z
    + C(year)
    + log_living_pop_z * C(cluster_type)
    + food_accom_ratio_z * C(cluster_type)
"""

m4 = smf.ols(m4_formula, data=reg_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gu"]}
)

print(m4.summary())

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.793
Method:                 Least Squares   F-statistic:                     838.9
Date:                Thu, 07 May 2026   Prob (F-statistic):           2.56e-29
Time:                        10:52:05   Log-Likelihood:                 41.982
No. Observations:                 100   AIC:                            -43.96
Df Residuals:                      80   BIC:                             8.139
Df Model:                          19                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [12]:
model_compare = pd.DataFrame({
    "model": [
        "M1_basic",
        "M2_add_cluster",
        "M3_living_interaction",
        "M4_living_food_interaction"
    ],
    "nobs": [m1.nobs, m2.nobs, m3.nobs, m4.nobs],
    "r2": [m1.rsquared, m2.rsquared, m3.rsquared, m4.rsquared],
    "adj_r2": [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj, m4.rsquared_adj],
    "aic": [m1.aic, m2.aic, m3.aic, m4.aic],
    "bic": [m1.bic, m2.bic, m3.bic, m4.bic]
})

model_compare

,model,nobs,r2,adj_r2,aic,bic
0,M1_basic,100.0,0.602948,0.558336,24.273383,52.930255
1,M2_add_cluster,100.0,0.684454,0.636755,7.297212,43.769595
2,M3_living_interaction,100.0,0.739667,0.689483,-5.937303,38.350591
3,M4_living_food_interaction,100.0,0.832382,0.792573,-43.964658,8.138746


In [13]:
# 강서구 제외 분석
reg_no_gangseo = reg_data[reg_data["gu"] != "강서구"].copy()

m4_no_gangseo = smf.ols(m4_formula, data=reg_no_gangseo).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_no_gangseo["gu"]}
)

sensitivity = pd.DataFrame({
    "model": ["강서구 포함 모델", "강서구 없는 모델"],
    "nobs": [m4.nobs, m4_no_gangseo.nobs],
    "r2": [m4.rsquared, m4_no_gangseo.rsquared],
    "adj_r2": [m4.rsquared_adj, m4_no_gangseo.rsquared_adj],
    "aic": [m4.aic, m4_no_gangseo.aic],
    "bic": [m4.bic, m4_no_gangseo.bic]
})

sensitivity

,model,nobs,r2,adj_r2,aic,bic
0,강서구 포함 모델,100.0,0.832382,0.792573,-43.964658,8.138746
1,강서구 없는 모델,96.0,0.854946,0.818683,-108.988701,-57.701737


In [14]:
# interaction 계수 방향 보기
coef_compare = pd.DataFrame({
    "coef_all": m4.params,
    "p_all": m4.pvalues,
    "coef_no_gangseo": m4_no_gangseo.params,
    "p_no_gangseo": m4_no_gangseo.pvalues
})

coef_compare["coef_diff"] = coef_compare["coef_no_gangseo"] - coef_compare["coef_all"]

coef_compare

,coef_all,p_all,coef_no_gangseo,p_no_gangseo,coef_diff
Intercept,11.657426,0.000000,11.666774,0.000000e+00,0.009348
C(year)[T.2021],0.024495,0.715127,-0.027432,4.391202e-01,-0.051928
C(year)[T.2022],0.051532,0.420871,0.029852,5.790830e-01,-0.021680
C(year)[T.2023],0.044641,0.599445,-0.006180,9.250812e-01,-0.050821
C(cluster_type)[T.상업형],0.073175,0.396867,0.083366,2.407935e-01,0.010191
C(cluster_type)[T.유동집중형],0.053368,0.426084,0.109156,4.050217e-03,0.055788
C(cluster_type)[T.혼합형],0.351480,0.000095,0.124247,3.087796e-03,-0.227233
log_total_pop_z,0.211684,0.294421,0.519065,2.476618e-05,0.307380
elderly_ratio_z,-0.027277,0.572068,0.035860,1.513191e-01,0.063137
day_night_ratio_z,0.195405,0.226950,0.334077,6.423201e-03,0.138671


In [15]:
# robust covariance가 아닌 일반 OLS 기준 nested F-test용
m1_plain = smf.ols(m1_formula, data=reg_data).fit()
m2_plain = smf.ols(m2_formula, data=reg_data).fit()
m3_plain = smf.ols(m3_formula, data=reg_data).fit()
m4_plain = smf.ols(m4_formula, data=reg_data).fit()

print("M1 vs M2:", m2_plain.compare_f_test(m1_plain))
print("M2 vs M3:", m3_plain.compare_f_test(m2_plain))
print("M3 vs M4:", m4_plain.compare_f_test(m3_plain))

M1 vs M2: (7.404603635575735, 0.00018046329134539815, 3.0)
M2 vs M3: (5.867790058943603, 0.0011029375872370912, 3.0)
M3 vs M4: (14.75018724486331, 9.86913614363026e-08, 3.0)


In [16]:
# 1. 강서구 제외 데이터로 기준모형 적합
baseline_data = reg_data[reg_data["gu"] != "강서구"].copy()

baseline_model = smf.ols(m4_formula, data=baseline_data).fit(
    cov_type="cluster",
    cov_kwds={"groups": baseline_data["gu"]}
)

print(baseline_model.summary())
# 2. 전체 데이터에 대해 예측
reg_data_pred = reg_data.copy()

reg_data_pred["pred_log_waste"] = baseline_model.predict(reg_data_pred)

# 로그값을 원래 폐기물 총량 단위로 변환
reg_data_pred["pred_waste_total"] = np.exp(reg_data_pred["pred_log_waste"])

# 잔차
reg_data_pred["residual"] = reg_data_pred["waste_total"] - reg_data_pred["pred_waste_total"]

# 로그 잔차: 비율 해석에 좋음
reg_data_pred["log_residual"] = (
    reg_data_pred["log_waste_total"] - reg_data_pred["pred_log_waste"]
)

# 실제가 기대보다 몇 % 높은지
reg_data_pred["residual_pct"] = (np.exp(reg_data_pred["log_residual"]) - 1) * 100

reg_data_pred[[
    "year", "gu", "cluster_type",
    "waste_total", "pred_waste_total",
    "residual", "residual_pct"
]].sort_values("residual_pct", ascending=False).head(15)

                            OLS Regression Results                            
Dep. Variable:        log_waste_total   R-squared:                       0.855
Model:                            OLS   Adj. R-squared:                  0.819
Method:                 Least Squares   F-statistic:                     828.0
Date:                Thu, 07 May 2026   Prob (F-statistic):           3.62e-28
Time:                        10:52:05   Log-Likelihood:                 74.494
No. Observations:                  96   AIC:                            -109.0
Df Residuals:                      76   BIC:                            -57.70
Df Model:                          19                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,year,gu,cluster_type,waste_total,pred_waste_total,residual,residual_pct
28,2021,강서구,혼합형,751795,186118.677824,565676.322176,303.933130
53,2022,강서구,혼합형,585756,194532.720105,391223.279895,201.109243
78,2023,강서구,혼합형,490315,189532.915866,300782.084134,158.696490
70,2022,용산구,상업형,216363,141552.045035,74810.954965,52.850494
95,2023,용산구,상업형,208742,141067.234686,67674.765314,47.973412
37,2021,마포구,혼합형,199008,146534.207689,52473.792311,35.809927
10,2020,동대문구,유동집중형,161082,124188.102082,36893.897918,29.708078
81,2023,구로구,혼합형,171610,146484.403950,25125.596050,17.152404
67,2022,송파구,혼합형,301830,264173.162372,37656.837628,14.254604
54,2022,관악구,주거형,161574,141504.835878,20069.164122,14.182670


In [17]:
resid_2023 = reg_data_pred[reg_data_pred["year"] == 2023].copy()

resid_2023[[
    "gu", "cluster_type",
    "waste_total", "pred_waste_total",
    "residual", "residual_pct"
]].sort_values("residual_pct", ascending=False)

,gu,cluster_type,waste_total,pred_waste_total,residual,residual_pct
78,강서구,혼합형,490315,189532.915866,300782.084134,158.696490
95,용산구,상업형,208742,141067.234686,67674.765314,47.973412
81,구로구,혼합형,171610,146484.403950,25125.596050,17.152404
79,관악구,주거형,146875,134653.402259,12221.597741,9.076338
82,금천구,혼합형,101244,93330.191137,7913.808863,8.479366
84,도봉구,주거형,102431,94439.922858,7991.077142,8.461546
97,종로구,상업형,97419,91636.360281,5782.639719,6.310421
77,강북구,유동집중형,88886,86810.301421,2075.698579,2.391074
92,송파구,혼합형,262182,259877.101937,2304.898063,0.886918
94,영등포구,상업형,166688,167798.716831,-1110.716831,-0.661934
